In [1]:
# INSTALL MONGODB
!sudo apt-get install gnupg curl
!curl -fsSL https://pgp.mongodb.com/server-8.0.asc | sudo gpg -o /usr/share/keyrings/mongodb-server-8.0.gpg --dearmor
!echo "deb [ arch=amd64,arm64 signed-by=/usr/share/keyrings/mongodb-server-8.0.gpg ] https://repo.mongodb.org/apt/ubuntu jammy/mongodb-org/8.0 multiverse" | sudo tee /etc/apt/sources.list.d/mongodb-org-8.0.list
!sudo apt-get update
!sudo apt install -y mongodb-org

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
curl is already the newest version (7.81.0-1ubuntu1.23).
gnupg is already the newest version (2.2.27-3ubuntu2.5).
gnupg set to manually installed.
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.
deb [ arch=amd64,arm64 signed-by=/usr/share/keyrings/mongodb-server-8.0.gpg ] https://repo.mongodb.org/apt/ubuntu jammy/mongodb-org/8.0 multiverse
Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://repo.mongodb.org/apt/ubuntu jammy/mongodb-org/8.0 InRelease [3,005 B]
Get:4 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:7 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [89.0 kB]
Get:8 http://security.ubuntu.co

In [2]:
# RUN MONGODB SERVER
# Create the data directory (required by MongoDB)
!mkdir -p /data/db

# Start MongoDB in the background
import os
os.system("mongod --fork --logpath /var/log/mongodb/mongod.log --dbpath /data/db")

0

In [20]:
# TEST MONGODB
!pip install pymongo

from pymongo import MongoClient

# Connect to the local instance
client = MongoClient("mongodb://localhost:27017/")
db = client.test_database
mongodb_collection = db.test_collection

# Insert a sample document
mongodb_collection.insert_one({"name": "Colab User", "status": "Connected"})
print(mongodb_collection.find_one())
mongodb_collection.delete_many({})

{'_id': ObjectId('69ef0edf402eea57166f689e'), 'name': 'Colab User', 'status': 'Connected'}


DeleteResult({'n': 1, 'ok': 1.0}, acknowledged=True)

In [8]:
# INSTALL REDIS IN-MEMORY DB
!curl -fsSL https://packages.redis.io/redis-stack/redis-stack-server-6.2.6-v7.jammy.x86_64.tar.gz -o redis-stack-server.tar.gz
!tar -xvf redis-stack-server.tar.gz
!pip install redis

./
./redis-stack-server-6.2.6-v7/
./redis-stack-server-6.2.6-v7/bin/
./redis-stack-server-6.2.6-v7/bin/redis-benchmark
./redis-stack-server-6.2.6-v7/bin/redis-cli
./redis-stack-server-6.2.6-v7/bin/redis-sentinel
./redis-stack-server-6.2.6-v7/bin/redis-stack-server
./redis-stack-server-6.2.6-v7/bin/redis-check-rdb
./redis-stack-server-6.2.6-v7/bin/redis-check-aof
./redis-stack-server-6.2.6-v7/bin/redis-server
./redis-stack-server-6.2.6-v7/share/
./redis-stack-server-6.2.6-v7/share/RSAL_LICENSE
./redis-stack-server-6.2.6-v7/share/APACHE_LICENSE
./redis-stack-server-6.2.6-v7/lib/
./redis-stack-server-6.2.6-v7/lib/redisgraph.so
./redis-stack-server-6.2.6-v7/lib/redistimeseries.so
./redis-stack-server-6.2.6-v7/lib/rejson.so
./redis-stack-server-6.2.6-v7/lib/redisbloom.so
./redis-stack-server-6.2.6-v7/lib/redisearch.so
./redis-stack-server-6.2.6-v7/etc/
./redis-stack-server-6.2.6-v7/etc/README
./redis-stack-server-6.2.6-v7/etc/redis-stack.conf
./redis-stack-server-6.2.6-v7/etc/redis-stack-se

In [9]:
# RUN REDIS SERVER
!./redis-stack-server-6.2.6-v7/bin/redis-stack-server --daemonize yes

Starting redis-stack-server, database path ./redis-stack-server-6.2.6-v7/var/db/redis-stack


In [18]:
# TES REDIS SERVER
import redis

redis_client = redis.Redis(host = 'localhost', port=6379)

print("Is REDIS running: {}".format(redis_client.ping()))

redis_client.set('foo', 'bar')
redis_client.get('foo')
redis_client.delete('foo')

Is REDIS running: True


1

In [5]:
# DOWNLOAD DATASET
import kagglehub

# Replace with your desired path
os.environ['KAGGLEHUB_CACHE'] = '/content/kaggle'

#path = kagglehub.dataset_download("shrashtisinghal/mongo-db-datsets")
path = kagglehub.dataset_download("otto/recsys-dataset")

print("Path to dataset files:", path)

jsonl_file = path + "/otto-recsys-test.jsonl"

Using Colab cache for faster access to the 'recsys-dataset' dataset.
Path to dataset files: /kaggle/input/recsys-dataset


In [19]:
# A class implementing write-through

class WriteThroughImpl:
    def __init__(self, redis_client, mongodb_collection):
        #self.mem_db = redis.Redis(host = 'localhost', port=6379)
        self.mem_db = redis_client
        self.disk_db = mongodb_collection
        #self.disk_db = db.test_collection

    def write(self, key, event_data):
        success = True
        # Update in-memory cache if possible
        try:
          self.mem_db.set(key, event_data)
        except redis.ResponseError as e:
          if "OOM" in str(e):
            print("Redis is Out of Memory!")
          else:
            print("Writing to Redis failed!")
          success = False

        # Immediately persist to disk (example using mongodb)
        self.disk_db.insert_one({key: event_data})

        return success

#!mongoimport --db test_database --collection test_collection --file {json_file} --jsonArray
#!mongoimport --db test_database --collection test_collection --file {jsonl_file}


In [49]:
# Test write-through

import pandas as pd
import math
import json
import tqdm

# Menggunakan iterator karena data terlalu besar untuk dimuat di memori
reader = pd.read_json(jsonl_file, lines=True, chunksize=1000)

# Buat writer dari kelas yang sudah dibuat
writer = WriteThroughImpl(redis_client, mongodb_collection)

# Check in-memory limit
#max_memory = redis_client.config_get('maxmemory').get('maxmemory')
#print(f"Redis in-memory limit: {max_memory} bytes")

# Kosongkan database
redis_client.flushdb()
mongodb_collection.delete_many({})

# Looping setiap chunk

with tqdm.tqdm(unit="rec") as pbar:
  # Iterasi semua record
  for chunk in reader:
    # Tulis setiap row sebagai satu record
    for index, row in chunk.iterrows():
      key, event_data = str(row['session']), json.dumps(row['events'])
      ret = writer.write(key, event_data)
      if not ret:
        break

    pbar.update(len(chunk))

# Cek apakah database sudah berisi data
print("\nJumlah records in-mem: {}, on-disk: {}".format(
    redis_client.dbsize(),mongodb_collection.count_documents({})))

1671803rec [00:42, 39762.52rec/s]


Jumlah records, in-mem: 1672, on-dis: 1672


In [71]:
# TEST SEARCHING A KEY

# cursor, keys = redis_client.scan(cursor=0, count=10)
# for key in keys:
#     print(key, redis_client.get(key))
search_key = "14282779"
print("In-mem: ", key, redis_client.get(search_key))

#for doc in mongodb_collection.find().limit(10):
#  print(doc)
print("On-disk: ", mongodb_collection.find_one({key: {"$exists": True}}))

In-mem:  14282779 b'[{"aid": 594986, "ts": 1662228662130, "type": "clicks"}, {"aid": 594986, "ts": 1662228958503, "type": "clicks"}, {"aid": 1376814, "ts": 1662229909941, "type": "clicks"}, {"aid": 923438, "ts": 1662230075282, "type": "clicks"}, {"aid": 821533, "ts": 1662232798993, "type": "clicks"}, {"aid": 1841413, "ts": 1662233576460, "type": "clicks"}]'
On-disk:  {'_id': ObjectId('69ef2617402eea571670076b'), '14282779': '[{"aid": 594986, "ts": 1662228662130, "type": "clicks"}, {"aid": 594986, "ts": 1662228958503, "type": "clicks"}, {"aid": 1376814, "ts": 1662229909941, "type": "clicks"}, {"aid": 923438, "ts": 1662230075282, "type": "clicks"}, {"aid": 821533, "ts": 1662232798993, "type": "clicks"}, {"aid": 1841413, "ts": 1662233576460, "type": "clicks"}]'}
